# 1회차 실습: 2R Robot FK / IK

이 노트북은 학생 배포용 실습 자료입니다.

이번 시간의 목표는 2R 로봇을 예제로 해서, 관절각과 말단 위치의 관계를 직접 구현하고 확인하는 것입니다.

이번 노트북에서 제공되는 것:
- 기본 helper 함수와 plotting 코드
- 2R 로봇 클래스 뼈대
- 결과를 바로 확인할 수 있는 검산 / 시각화 셀

이번 노트북에서 직접 구현할 것:
- `Planar2RRobot.fk()`
- `analytic_ik_2r()`
- `sample_workspace()`

실습 방법:
1. 위에서부터 셀을 순서대로 실행합니다.
2. `TODO(student)`가 있는 함수만 직접 구현합니다.
3. 구현 후 바로 아래 검산 셀에서 결과를 확인합니다.


## 오늘 할 일

이번 시간에는 아래 네 가지를 순서대로 확인합니다.

1. 관절각이 바뀌면 로봇 팔 모양이 어떻게 달라지는지 보기
2. FK를 구현해서 말단 위치 계산하기
3. IK를 구현해서 목표 위치에 대응하는 관절각 구하기
4. reachable / unreachable target을 구분하기

핵심은 식을 외우는 것보다, `각도 변화가 위치 변화로 어떻게 이어지는지` 감을 잡는 것입니다.


In [ ]:
# helper functions
# don't modify this block!

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
try:
    from IPython.display import HTML
except ImportError:
    def HTML(x):
        return x

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['animation.embed_limit'] = 30


def wrap_to_pi(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi


def deg2rad(q):
    return np.deg2rad(q)


def rad2deg(q):
    return np.rad2deg(q)


def plot_robot(robot, q, ax=None, target_position=None, title=None, color='C0'):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))

    q = np.asarray(q, dtype=float)
    positions = robot.joint_positions(q)
    ee_pose = robot.fk(q)

    ax.plot(positions[:, 0], positions[:, 1], '-o', linewidth=3, markersize=8, color=color)
    ax.scatter([0], [0], s=120, marker='s', color='k')

    x, y, phi = ee_pose
    arrow_len = 0.20
    ax.arrow(x, y, arrow_len * np.cos(phi), arrow_len * np.sin(phi), head_width=0.05, length_includes_head=True, color=color)

    if target_position is not None:
        tx, ty = target_position
        ax.scatter([tx], [ty], s=120, marker='x', color='red')

    if title is not None:
        ax.set_title(title)

    reach = np.sum(robot.link_lengths) + 0.25
    ax.set_xlim(-reach, reach)
    ax.set_ylim(-reach, reach)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    return ax


## 1. 2R Robot Model

먼저 오늘 사용할 2R 평면 로봇 모델을 정의합니다.

- joint angle: $q = [q_1, q_2]^T$
- link length: $l_1, l_2$
- end-effector pose: $[x, y, \phi]$

계산의 기준이 되는 식은 아래 세 개입니다.

$
x = l_1\cos q_1 + l_2\cos(q_1 + q_2)
$

$
y = l_1\sin q_1 + l_2\sin(q_1 + q_2)
$

$
\phi = q_1 + q_2
$

식 자체를 외우는 것보다,
`각도가 누적되면서 링크 끝점이 어디로 가는지`를 머릿속으로 그릴 수 있으면 충분합니다.


### Implement: FK 함수 완성하기

이 셀에서는 `Planar2RRobot.fk()`를 완성합니다.

여기서 해야 할 일은 단순합니다.

- 입력은 `q = [q1, q2]`
- 출력은 `np.array([x, y, phi])`
- 두 번째 링크는 항상 `q1 + q2` 방향을 봅니다.
- 마지막 각도 `phi`는 `wrap_to_pi()`로 정리합니다.

이미 `joint_positions()`는 주어져 있으니,
이번에는 말단 pose 계산만 직접 채워 보면 됩니다.


In [ ]:
class Planar2RRobot:
    def __init__(self, link_lengths=(1.0, 0.8), joint_limits=None):
        self.link_lengths = np.asarray(link_lengths, dtype=float)
        if joint_limits is None:
            joint_limits = np.array([
                [-np.pi, np.pi],
                [-np.pi, np.pi],
            ], dtype=float)
        self.joint_limits = np.asarray(joint_limits, dtype=float)

    def is_within_joint_limits(self, q, atol=1e-9):
        q = wrap_to_pi(np.asarray(q, dtype=float))
        lower = self.joint_limits[:, 0]
        upper = self.joint_limits[:, 1]
        return np.all(q >= lower - atol) and np.all(q <= upper + atol)

    def joint_positions(self, q):
        q = np.asarray(q, dtype=float)
        l1, l2 = self.link_lengths
        q1, q2 = q
        p0 = np.array([0.0, 0.0])
        p1 = p0 + l1 * np.array([np.cos(q1), np.sin(q1)])
        p2 = p1 + l2 * np.array([np.cos(q1 + q2), np.sin(q1 + q2)])
        return np.vstack([p0, p1, p2])

    def fk(self, q):
        q = np.asarray(q, dtype=float)
        l1, l2 = self.link_lengths
        q1, q2 = q

        # TODO(student): 아래 3줄을 직접 구현해 보세요.
        # q12 = ...
        # x = ...
        # y = ...
        # phi = ...


robot = Planar2RRobot(link_lengths=(1.0, 0.8))
print('Link lengths:', robot.link_lengths)
print('Joint limits [deg]:')
print(rad2deg(robot.joint_limits))


## 2. Check FK implementation

FK가 잘 구현되었다면, 관절각을 바꿨을 때 말단 위치가 어떻게 달라지는지 바로 볼 수 있습니다.


In [ ]:
q = deg2rad([35, 50])
pose = robot.fk(q)
positions = robot.joint_positions(q)

print('q [deg] =', rad2deg(q))
print(f'end-effector pose [x, y, phi(deg)] = [{pose[0]:.4f}, {pose[1]:.4f}, {rad2deg(pose[2]):.2f}]')
print('joint positions =')
print(positions)

fig, ax = plt.subplots(figsize=(6, 6))
plot_robot(robot, q, ax=ax, title='FK check: one robot pose')
plt.show()


## 3. 움직이는 데모: FK만으로 만드는 simple 2R Robot movement

관절각을 시간에 따라 바꾸면 FK가 매 순간 로봇 자세를 계산합니다.
아래 셀은 단순한 사인파 관절 운동만으로 말단의 궤적을 만드는 예시입니다.


In [ ]:
# simple movement
sim_t = np.linspace(0, 2 * np.pi, 90)
dance_q = np.c_[deg2rad(35) + 0.9 * np.sin(sim_t), deg2rad(55) + 0.8 * np.sin(2 * sim_t + 0.6)]
dance_points = np.array([robot.joint_positions(qi) for qi in dance_q])
end_trace = dance_points[:, -1, :]

fig, ax = plt.subplots(figsize=(5, 5))
line, = ax.plot([], [], 'o-', linewidth=3, markersize=8, color='tab:blue')
trace_line, = ax.plot([], [], '--', linewidth=2, color='tab:orange', alpha=0.8)
time_text = ax.text(0.03, 0.95, '', transform=ax.transAxes)
reach = np.sum(robot.link_lengths) + 0.25
ax.set_xlim(-reach, reach)
ax.set_ylim(-reach, reach)
ax.set_aspect('equal', adjustable='box')
ax.grid(True)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('2R robot dance from FK')


def update_dance(k):
    pts = dance_points[k]
    line.set_data(pts[:, 0], pts[:, 1])
    trace_line.set_data(end_trace[: k + 1, 0], end_trace[: k + 1, 1])
    time_text.set_text(f'frame {k:02d}')
    return line, trace_line, time_text


anim = FuncAnimation(fig, update_dance, frames=len(dance_q), interval=55, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())


## 4. IK: 목표점으로 가는 관절각 찾기

이제 반대로 생각해 보겠습니다.

지금까지는 관절각 $q$를 알고 있을 때 말단 위치를 계산했습니다.
이번에는 목표 위치 $(x, y)$가 주어졌을 때, 그 위치로 가는 관절각을 구해 보겠습니다.

Inverse Kinematics 과정은 아래와 같다.

1. 목표점까지의 거리로 $c_2$를 계산한다.
2. $q_2$는 보통 두 개의 branch를 가진다.
3. 그 branch에 따라 $q_1$도 달라진다.


### Analytic IK 함수 완성하기

![alt text](figs/week1_ik.png)

구체적으로는 아래와 같은 과정으로 구현하면 됨

1. 목표점으로부터 $r^2 = x^2 + y^2$ 계산
2. $c_2$ 계산 후 도달 가능 여부 확인
3. $q_2$의 두 branch 계산
4. 각 branch마다 $q_1$ 계산



In [ ]:
def analytic_ik_2r(robot, target_position, check_joint_limits=True, eps=1e-9):
    x, y = np.asarray(target_position, dtype=float)
    l1, l2 = robot.link_lengths

    # TODO(student): 아래 reachability / IK 계산을 직접 채워 보세요.
    # 1. r2 계산
    # 2. c2 계산
    # 3. reachable 여부를 True / False로 판단
    # 4. 두 branch에 대해 q1, q2 계산
    # 5. 최종적으로 (solutions, reachable) 반환
    raise NotImplementedError('Implement analytic_ik_2r() before running the next cells.')


## 5. 같은 목표, 다른 팔 모양

아래 목표점은 두 branch가 모두 존재하는 예시입니다.

직전 block에서 구현한 analytic ik가 잘 작동하는지 확인해보고, 하나의 target point에 대해 2개의 해가 구해짐을 시각적으로 확인해보세요.


In [ ]:
target_position = np.array([1.1, 0.7])
solutions, reachable = analytic_ik_2r(robot, target_position)

print('Target position [x, y] =', target_position)
print('Reachable =', reachable)
print('number of solutions =', len(solutions))

for i, sol in enumerate(solutions):
    fk_sol = robot.fk(sol)
    err = fk_sol[:2] - target_position
    print()
    print(f'Solution {i}')
    print('q [deg] =', rad2deg(sol))
    print('position error =', err)

fig, axes = plt.subplots(1, max(1, len(solutions)), figsize=(6 * max(1, len(solutions)), 5))
if len(solutions) == 1:
    axes = [axes]

for i, sol in enumerate(solutions):
    plot_robot(robot, sol, ax=axes[i], target_position=target_position, title=f'IK branch {i}')

plt.show()


## 6. 움직이는 데모: 두 branch가 같은 목표 경로를 따라가면?

같은 원형 target path를 따라가더라도 branch가 다르면 팔의 움직임이 꽤 다르게 보입니다.
이 셀은 `IK 해의 연속성`이 왜 중요한지 감으로 보여줍니다.


In [ ]:
race_center = np.array([0.95, 0.35])
race_radius = 0.18
race_angles = np.linspace(0, 2 * np.pi, 90, endpoint=False)
race_targets = race_center + race_radius * np.c_[np.cos(race_angles), np.sin(race_angles)]

race_q0, race_q1, valid_targets = [], [], []
for p in race_targets:
    sols = analytic_ik_2r(robot, p)[0]
    if len(sols) == 2:
        race_q0.append(sols[0])
        race_q1.append(sols[1])
        valid_targets.append(p)

race_q0 = np.array(race_q0)
race_q1 = np.array(race_q1)
valid_targets = np.array(valid_targets)
race_pts0 = np.array([robot.joint_positions(qi) for qi in race_q0])
race_pts1 = np.array([robot.joint_positions(qi) for qi in race_q1])

fig, ax = plt.subplots(figsize=(6, 6))
arm0, = ax.plot([], [], 'o-', linewidth=3, color='tab:blue', label='branch 0')
arm1, = ax.plot([], [], 'o-', linewidth=3, color='tab:orange', label='branch 1')
target_dot, = ax.plot([], [], 'r*', markersize=12, label='target')
ax.plot(valid_targets[:, 0], valid_targets[:, 1], 'k--', alpha=0.35, label='target path')
reach = np.sum(robot.link_lengths) + 0.25
ax.set_xlim(-reach, reach)
ax.set_ylim(-reach, reach)
ax.set_aspect('equal', adjustable='box')
ax.grid(True)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('two IK branches following the same target')
ax.legend(loc='upper right')


def update_race(k):
    arm0.set_data(race_pts0[k, :, 0], race_pts0[k, :, 1])
    arm1.set_data(race_pts1[k, :, 0], race_pts1[k, :, 1])
    target_dot.set_data([valid_targets[k, 0]], [valid_targets[k, 1]])
    return arm0, arm1, target_dot


anim = FuncAnimation(fig, update_race, frames=len(valid_targets), interval=60, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())


## 7. Workspace 샘플링

이번에는 로봇이 대략 어디까지 닿을 수 있는지 그림으로 확인해 보겠습니다.

방법은 단순합니다.

- joint angle을 많이 찍어 본다.
- 각 joint angle에 대해 FK를 계산한다.
- end-effector 위치만 모아서 뿌려 본다.

이렇게 하면 도달 가능한 영역의 전체 모양을 아주 직관적으로 볼 수 있습니다.


### 구현 포인트: Workspace 샘플링 함수 완성하기

이 셀에서 학생들이 할 일은 많지 않습니다.

- 무작위 joint angle 샘플 `qs`는 이미 준비되어 있음
- 각 `q`마다 `robot.fk(q)`를 호출
- 결과를 모아서 `poses` 배열로 만들기

이 구현은 간단하지만,
앞에서 만든 FK 함수가 정말 제대로 동작하는지 한 번 더 확인하는 좋은 체크포인트입니다.


In [ ]:
def sample_workspace(robot, num_samples=12000):
    rng = np.random.default_rng(0)
    lower = robot.joint_limits[:, 0]
    upper = robot.joint_limits[:, 1]
    qs = rng.uniform(lower, upper, size=(num_samples, 2))

    # TODO(student): fk를 이용해 모든 q에 대한 end-effector pose를 구해 보세요.
    raise NotImplementedError('Implement sample_workspace() before running the next cells.')


qs, poses = sample_workspace(robot, num_samples=12000)

plt.figure(figsize=(6, 6))
plt.scatter(poses[:, 0], poses[:, 1], s=1, alpha=0.25)
plt.axis('equal')
plt.grid(True)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Sampled workspace of the 2R robot')
plt.show()


## 8. Reachable / Unreachable 한 번에 보기

이번에는 target만 보는 것이 아니라, 로봇이 실제로 닿을 수 있는 workspace를 같이 보겠습니다.

회색 점들은 joint를 무작위로 많이 샘플링해서 얻은 end-effector 위치입니다.
이 점구름 안쪽에 있는 target은 대체로 reachable하고, 바깥에 있는 target은 unreachable합니다.

같은 2R 로봇이라도, 그림으로 보면 `왜 안 닿는지`가 수식보다 훨씬 빠르게 보입니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

reachable_target = np.array([1.1, 0.7])
unreachable_target = np.array([2.2, 0.0])

reachable_solutions, reachable_flag = analytic_ik_2r(robot, reachable_target)
solutions_bad, reachable_bad = analytic_ik_2r(robot, unreachable_target)

workspace_xy = poses[:, :2]
reach = np.sum(robot.link_lengths) + 0.5

for ax, target, title in [
    (axes[0], reachable_target, 'reachable target inside workspace'),
    (axes[1], unreachable_target, 'unreachable target outside workspace'),
]:
    ax.scatter(workspace_xy[:, 0], workspace_xy[:, 1], s=1, alpha=0.12, color='0.6', label='sampled workspace')
    ax.plot(target[0], target[1], 'r*', markersize=20, label='target')
    # ax.set_xlim(-reach, reach)
    # ax.set_ylim(-reach, reach)
    # ax.set_aspect('equal', adjustable='box')
    ax.grid(True)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(title)

plot_robot(robot, reachable_solutions[0], ax=axes[0], target_position=reachable_target)
plot_robot(robot, deg2rad([0, 0]), ax=axes[1], target_position=unreachable_target)
axes[0].legend(loc='upper right')
axes[1].legend(loc='upper left')
axes[0].set_xlim(-reach, reach)
axes[0].set_ylim(-reach, reach)
axes[1].set_xlim(-reach, reach)
axes[1].set_ylim(-reach, reach)

plt.show()

print('reachable target =', reachable_target)
print('reachable target flag =', reachable_flag)
print('number of reachable solutions =', len(reachable_solutions))
print()
print('unreachable target =', unreachable_target)
print('unreachable target flag =', reachable_bad)
print('number of unreachable solutions =', len(solutions_bad))


## 9. Joint Limit과 해 선택

실제 로봇에서는 해가 여러 개 있어도 아무 해나 쓰지 않습니다.

보통은 다음 기준을 같이 봅니다.

- joint limit을 만족하는가?
- 현재 자세에서 너무 많이 움직이지 않는가?

이 부분은 앞으로 trajectory와 motion planning으로 갈 때도 계속 중요하게 등장합니다.


In [ ]:
def choose_closest_solution(solutions, q_current):
    if len(solutions) == 0:
        return None, None

    distances = []
    for q in solutions:
        dq = wrap_to_pi(q - q_current)
        distances.append(np.linalg.norm(dq))

    best_idx = int(np.argmin(distances))
    return solutions[best_idx], distances[best_idx]


limited_robot = Planar2RRobot(
    link_lengths=(1.0, 0.8),
    joint_limits=deg2rad(np.array([
        [-180, 180],
        [0, 110],
    ])),
)

q_current = deg2rad([15, 30])
solutions_unlimited, _ = analytic_ik_2r(robot, target_position)
solutions_limited, reachable_limited = analytic_ik_2r(limited_robot, target_position)
q_best, best_dist = choose_closest_solution(solutions_unlimited, q_current)

print('Current q [deg] =', rad2deg(q_current))
print('Unlimited solutions:')
for i, sol in enumerate(solutions_unlimited):
    dist = np.linalg.norm(wrap_to_pi(sol - q_current))
    print(f'  solution {i}: q={np.round(rad2deg(sol), 2)}, distance={dist:.4f}')

print()
print('Limited robot reachable =', reachable_limited)
print('Chosen solution [deg] =', np.round(rad2deg(q_best), 2))
print('Chosen distance =', best_dist)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_robot(robot, q_current, ax=axes[0], target_position=target_position, title='current configuration')
plot_robot(robot, q_best, ax=axes[1], target_position=target_position, title='closest IK solution')
plt.show()


## 제출 전 체크

아래 항목을 스스로 확인해 보세요.

- `fk()`가 올바른 `x, y, phi`를 반환하는가?
- `analytic_ik_2r()`가 reachable한 점에서 해를 반환하는가?
- unreachable한 점에서는 빈 해와 `reachable=False`를 반환하는가?
- workspace 시각화가 정상적으로 그려지는가?
